# Part 7b: Deployment on PYNQ-Z2
The following section is the code to execute in the pynq-z2 jupyter notebook to execute NN inference. 

The following cells are intended to run on a pynq-z2, they will not run on the server used to train and synthesize models!

First, import our driver `Overlay` class. We'll also load the test data.

In [ ]:
from axi_stream_driver import NeuralNetworkOverlay
import numpy as np

X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

Create a `NeuralNetworkOverlay` object. This will download the `Overlay` (bitfile) onto the PL of the pynq-z2. We provide the `X_test.shape` and `y_test.shape` to allocate some buffers for the data transfer.

In [ ]:
nn = NeuralNetworkOverlay('hls4ml_nn.bit', X_test.shape, y_test.shape)

Now run the prediction! When we set `profile=True` the function times the inference, and prints out a summary as well as returning the profiling information. We also save the output to a file so we can do some validation.

In [ ]:
y_hw, latency, throughput = nn.predict(X_test, profile=True)

An example print out looks like:

```
Classified 166000 samples in 0.402568 seconds (412352.6956936468 inferences / s)
```

Now let's save the output and transfer this back to the host.

In [ ]:
np.save('y_hw.npy', y_hw)

Now, go back to the host and follow `part7c_validation.ipynb`

## 📦 Data preservation & provenance with Dataerai

The cell below preserves **this notebook's** artifacts as versioned Dataerai assets in a **per-notebook collection**, links them into the shared lineage DAG (with verifiable DID citations), and adds a static **recording** — the notebook file plus its execution log — using the existing Dataerai *beta* APIs (no backend changes).

It is **idempotent** and safe to re-run; running the parts in order accumulates the full provenance graph into `PROVENANCE.md` / `provenance_manifest.json`. See **[DATAERAI_PROVENANCE.md](DATAERAI_PROVENANCE.md)** for one-time setup.

In [ ]:
# Dataerai — preserve this notebook's artifacts into a per-notebook collection,
# link the lineage DAG, and record the notebook + execution log.
# No-op unless `dataerai_hls4ml` is importable and DATAERAI_PROVENANCE != 0.
# Setup (see DATAERAI_PROVENANCE.md):
#   dataerai auth login --server https://beta.dataerai.com
#   export DATAERAI_PROJECT_ID=<your-project-uuid>
# Or run fully offline (records lineage locally, uploads nothing): export DATAERAI_DRY_RUN=1
try:
    import dataerai_hls4ml as dp
except ImportError:
    dp = None

if dp and dp.enabled():
    manifest = dp.capture(notebook="part7b_deployment")
    mode = 'dry-run' if manifest['dry_run'] else manifest['server']
    print(f"Dataerai provenance: run {manifest['run_id']} — "
          f"{len(manifest['artifacts'])} assets, {len(manifest['edges'])} edges ({mode})")
    print('Wrote PROVENANCE.md and provenance_manifest.json')